# Agent Live Preview: Public URL from the Sandbox

Run a tiny `python3 -m http.server` inside the sandbox, expose port 8000
via ACA's per-port public URL, and have an OpenAI agent announce the URL
to the user. The URL is gated by Entra ID - anyone signed in to your
tenant can hit it; random visitors are blocked.

## Prerequisites
- Azure CLI [signed in](https://learn.microsoft.com/cli/azure/authenticate-azure-cli-interactively)
- `azure-containerapps-sandbox` SDK (from GitHub Releases - see [lab README](./README.md#prerequisites)):
  ```bash
  gh release download v0.1.0b1 \
      --repo Azure-Samples/azure-container-apps-sandboxes \
      --pattern "azure_containerapps_sandbox-*.whl" \
      --output azure_containerapps_sandbox.whl
  pip install azure_containerapps_sandbox.whl
  pip install openai-agents
  ```
- An OpenAI provider, configured **once** for the whole lab in `provider_config.py`
  (see the [lab README](./README.md#prerequisites)). Azure OpenAI is preferred and
  uses Entra ID auth by default.

Click **Run All** or go **Step by Step**.

### 0. Initialize variables

In [ ]:
import os, json, shutil, subprocess, time
from azure.containerapps.sandbox import SandboxClient, SandboxGroupClient

lab_name = os.path.basename(os.path.dirname(os.path.abspath(
    globals().get('__vsc_ipynb_file__', os.path.join(os.getcwd(), '__file__')))))

# Resolve full path to the az CLI (on Windows it's az.cmd, which subprocess.run
# can't find by bare name without shell=True).
az = shutil.which('az')
if not az:
    raise RuntimeError(
        'Azure CLI not found on PATH. Install it from '
        'https://learn.microsoft.com/cli/azure/install-azure-cli and re-run.'
    )

account = json.loads(subprocess.run(
    [az, 'account', 'show', '-o', 'json'],
    capture_output=True, text=True, check=True).stdout)

subscription_id = account['id']
resource_group_name = f'lab-{lab_name}'
resource_group_location = 'westus2'
sandbox_group_name = f'{lab_name}-sg'
exposed_port = 8000

print(f'Lab:            {lab_name}')
print(f'Resource Group: {resource_group_name}')
print(f'Sandbox Group:  {sandbox_group_name}')
print(f'Exposed port:   {exposed_port}')

client = SandboxClient(subscription_id=subscription_id, resource_group=resource_group_name)
mgmt = SandboxGroupClient(subscription_id=subscription_id, resource_group=resource_group_name)

### 0a. Configure the OpenAI provider

Provider config is shared across the three notebooks in this lab. Copy [`provider_config.py.example`](./provider_config.py.example) to `provider_config.py` (gitignored) and fill in either the Azure OpenAI or OpenAI section -- all three notebooks pick it up automatically. If you prefer environment variables, leave it as-is and set `AZURE_OPENAI_ENDPOINT`/`AZURE_OPENAI_DEPLOYMENT` (Entra ID auth by default) or `OPENAI_API_KEY` in your shell before launching VS Code.

In [ ]:
import sys, os
_lab_dir = os.path.dirname(os.path.abspath(
    globals().get('__vsc_ipynb_file__', os.path.join(os.getcwd(), '__file__'))))
if _lab_dir not in sys.path:
    sys.path.insert(0, _lab_dir)

from provider import get_model
model = get_model()


### 1. Create the sandbox (`python-3.13` public image)


In [ ]:
!az group create --name {resource_group_name} --location {resource_group_location} -o none

group = mgmt.create_group(sandbox_group_name, location=resource_group_location)
print(f'Group: {group.name}')

# Ensure the signed-in user has data-plane access on the sandbox group.
# `mgmt.create_group` only needs ARM (control plane) rights, but
# `client.create_sandbox` calls the data plane (management.azuredevcompute.io)
# which requires a separate role on the sandbox group resource itself.
sg_id = group.id
data_role = 'Dev Compute SandboxGroup Data Owner'

signed_in_oid = subprocess.run(
    [az, 'ad', 'signed-in-user', 'show', '--query', 'id', '-o', 'tsv'],
    capture_output=True, text=True, check=True).stdout.strip()

existing = json.loads(subprocess.run(
    [az, 'role', 'assignment', 'list',
     '--assignee', signed_in_oid,
     '--scope', sg_id,
     '--role', data_role,
     '-o', 'json'],
    capture_output=True, text=True, check=True).stdout)

if not existing:
    print(f'Assigning "{data_role}" to {signed_in_oid} on sandbox group...')
    subprocess.run(
        [az, 'role', 'assignment', 'create',
         '--assignee-object-id', signed_in_oid,
         '--assignee-principal-type', 'User',
         '--role', data_role,
         '--scope', sg_id,
         '-o', 'none'],
        check=True)
    # Role assignments take a few seconds to propagate to the data plane.
    import time
    for i in range(30):
        time.sleep(5)
        try:
            sbx = client.create_sandbox(sandbox_group_name, disk='python-3.13')
            break
        except Exception as e:
            if '403' not in str(e) or i == 29:
                raise
            print(f'  waiting for role propagation... ({(i+1)*5}s)')
    else:
        raise RuntimeError('Role assignment did not propagate within 150s')
else:
    print(f'Role "{data_role}" already assigned.')
    sbx = client.create_sandbox(sandbox_group_name, disk='python-3.13')

sandbox_id = sbx.id
print(f'Sandbox: {sandbox_id}')

### 2. Write the page and start the server

In [ ]:
index_html = '''<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8" />
    <meta name="viewport" content="width=device-width, initial-scale=1.0" />
    <title>Live preview from an Azure sandbox</title>
    <style>
        body {
            font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif;
            max-width: 720px;
            margin: 4rem auto;
            padding: 0 1.25rem;
            color: #1f2933;
            line-height: 1.55;
        }
        h1 { color: #0078d4; margin-bottom: 0.25rem; }
        code { background: #f3f2f1; padding: 0.1rem 0.35rem; border-radius: 3px; }
        footer { margin-top: 3rem; color: #6b7280; font-size: 0.85rem; }
    </style>
</head>
<body>
    <h1>Hello from an Azure Container Apps sandbox</h1>
    <p>
        This page is being served by a tiny <code>python3 -m http.server</code>
        process running inside an ephemeral Azure sandbox. An OpenAI agent
        created the sandbox, mounted this file, launched the web server, and
        then resolved the public URL through ACA's per-port Entra-gated routing.
    </p>
    <p>
        The sandbox shuts down (and the URL stops working) as soon as the
        controlling notebook tears it down.
    </p>
    <footer>
        Demo asset for the Azure Container Apps Sandboxes labs.
    </footer>
</body>
</html>
'''

workspace = '/home/user/workspace'
client.exec(sandbox_id, sandbox_group_name, f'mkdir -p {workspace}')
client.write_file(sandbox_id, sandbox_group_name, f'{workspace}/index.html', index_html)
print(f'Wrote {workspace}/index.html')

start_cmd = (
    f'cd {workspace} && '
    f'nohup python3 -m http.server {exposed_port} '
    f'--directory {workspace} '
    f'>{workspace}/.http-server.log 2>&1 &'
)
client.exec(sandbox_id, sandbox_group_name, start_cmd)
print(f'Started python3 -m http.server on :{exposed_port}')

# Poll the port via Python's stdlib socket module -- works on every disk image
# (no dependency on ss/netstat/curl, which slim images may not ship).
probe_py = (
    'import socket, sys; '
    's = socket.socket(); s.settimeout(1); '
    f'sys.exit(0 if s.connect_ex(("127.0.0.1", {exposed_port})) == 0 else 1)'
)
probe_cmd = f"python3 -c '{probe_py}'"

for attempt in range(40):
    time.sleep(0.5)
    chk = client.exec(sandbox_id, sandbox_group_name, probe_cmd)
    if chk.exit_code == 0:
        print(f'Server is listening (attempt {attempt + 1})')
        break
else:
    print(f'WARNING: could not confirm http.server is listening on :{exposed_port}')
    log = client.exec(sandbox_id, sandbox_group_name, f'cat {workspace}/.http-server.log')
    print('--- http.server log ---')
    print(log.stdout or '(empty)')


### 3. Expose the port and grab the public URL

In [ ]:
port_info = client.add_port(sandbox_id, sandbox_group_name, exposed_port, anonymous=True)
url = port_info.url

if not url:
    raise RuntimeError(f'Public URL for :{exposed_port} not available')

print('=' * max(40, len(url) + 18))
print(f'  Live preview URL: {url}')
print('=' * max(40, len(url) + 18))
print('Open the URL in your browser. ACA will prompt you to sign in with')
print('the same Entra ID account you used for `az login`.')

### 4. Have the agent announce the URL

A trivial OpenAI agent that just announces the URL — proves the agent loop
works end-to-end on top of the sandbox infrastructure we just stood up.

In [ ]:
from agents import Agent, Runner

agent = Agent(
    name='Live preview announcer',
    model=model,
    instructions=(
        'Reply with exactly one short sentence that announces the URL '
        'the user provides. Do not call any tools.'
    ),
)

question = (
    f'I just launched a web server in a sandbox. Tell the user the URL is: {url}'
)
result = await Runner.run(agent, question)
print(f'assistant> {result.final_output}')

### 5. Clean up

In [ ]:
client.delete_sandbox(sandbox_id, sandbox_group_name)
print('Deleted sandbox (URL is now offline)')

mgmt.delete_group(sandbox_group_name)
print('Deleted group')

!az group delete --name {resource_group_name} --yes --no-wait
print('Deleting resource group (async)')